# Export Model to ONNX

Convert the fine-tuned RT-DETR model (`kaya-go/moku-v1`) to ONNX format for browser inference via ONNX Runtime WebAssembly.

**Sections:**

1. Load model from HF Hub
2. Export to ONNX (dynamic batch, opset 17)
3. Verify ONNX model (metadata + output comparison)
4. Benchmark ONNX vs PyTorch throughput
5. Push ONNX model to HF Hub

**Model outputs (raw, no post-processing baked in):**

- `logits`: `(batch, 300, num_classes)` — apply sigmoid to get per-class scores
- `pred_boxes`: `(batch, 300, 4)` — normalized `[cx, cy, w, h]` in `[0, 1]`

Post-processing (score threshold + top-4 corner selection) is handled downstream in JavaScript.


In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import onnx
import onnxruntime as ort
import torch
from huggingface_hub import HfApi
from transformers import RTDetrForObjectDetection

from moku.dataset import CATEGORIES, ID_TO_CATEGORY
from moku.model import load_image_processor

ONNX_PATH = Path("../artifacts/moku-v2.onnx")
ONNX_PATH.parent.mkdir(parents=True, exist_ok=True)
OPSET = 18

HF_MODEL = "kaya-go/moku-v2"

## Load Model from HF Hub

Load the fine-tuned model on CPU — ONNX export is always done on CPU.


In [2]:
image_processor = load_image_processor()

model = RTDetrForObjectDetection.from_pretrained(
    HF_MODEL,
    num_labels=len(CATEGORIES),
    id2label=ID_TO_CATEGORY,
    label2id=CATEGORIES,
    attn_implementation="eager",
)
model.eval()
model.cpu()

# Input size expected by RT-DETR r18vd
INPUT_H, INPUT_W = image_processor.size["height"], image_processor.size["width"]
print(f"Model: {HF_MODEL}")
print(f"Input size: {INPUT_H}x{INPUT_W}")
print(f"Num classes: {len(CATEGORIES)}  |  Num queries: {model.config.num_queries}")


Loading weights:   0%|          | 0/526 [00:00<?, ?it/s]

Model: kaya-go/moku-v2
Input size: 640x640
Num classes: 3  |  Num queries: 300


## Export to ONNX

A thin wrapper strips the HF output dataclass down to a plain `(logits, pred_boxes)` tuple, which ONNX requires. Dynamic axes on `batch_size` keep the graph flexible.


In [3]:
class _RTDetrOnnxWrapper(torch.nn.Module):
    """Strip HF output dataclass to a plain (logits, pred_boxes) tuple."""

    def __init__(self, model: RTDetrForObjectDetection) -> None:
        super().__init__()
        self.model = model

    def forward(self, pixel_values: torch.Tensor):
        out = self.model(pixel_values=pixel_values)
        return out.logits, out.pred_boxes


wrapper = _RTDetrOnnxWrapper(model)
wrapper.eval()

dummy = torch.randn(1, 3, INPUT_H, INPUT_W)

torch.onnx.export(
    wrapper,
    (dummy,),
    str(ONNX_PATH),
    opset_version=OPSET,
    input_names=["pixel_values"],
    output_names=["logits", "pred_boxes"],
    dynamic_axes={
        "pixel_values": {0: "batch_size"},
        "logits": {0: "batch_size"},
        "pred_boxes": {0: "batch_size"},
    },
    do_constant_folding=True,
    dynamo=False,
)

size_mb = ONNX_PATH.stat().st_size / 1e6
print(f"Exported → {ONNX_PATH}  ({size_mb:.1f} MB)")

/var/folders/fk/ty303kmn61x3y013qnk89g3h0000gn/T/ipykernel_73181/2679666677.py:18: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/Users/hadim/Code/libs/moku/.pixi/envs/default/lib/python3.12/site-packages/transformers/models/rt_detr/modeling_rt_detr_resnet.py:92: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if num_channels != self.num_channels:
/Users/hadim/Code/libs/moku/.pixi/envs/default/lib/python3.12/site-packages/

Exported → ../artifacts/moku-v2.onnx  (80.8 MB)


## Verify ONNX Model

Check graph metadata and confirm outputs match the PyTorch reference values (within float32 tolerance).


In [4]:
sess_options = ort.SessionOptions()
sess = ort.InferenceSession(str(ONNX_PATH), sess_options, providers=["CPUExecutionProvider"])

# Print graph metadata
print("=== Inputs ===")
for inp in sess.get_inputs():
    print(f"  {inp.name}: {inp.shape}  ({inp.type})")

print("\n=== Outputs ===")
for out in sess.get_outputs():
    print(f"  {out.name}: {out.shape}  ({out.type})")


=== Inputs ===
  pixel_values: ['batch_size', 3, 640, 640]  (tensor(float))

=== Outputs ===
  logits: ['batch_size', 'Gatherlogits_dim_1', 3]  (tensor(float))
  pred_boxes: ['batch_size', 'Gatherpred_boxes_dim_1', 4]  (tensor(float))


In [5]:
# Compare PyTorch vs ONNX outputs on a fixed random input.
# Note: topk is non-deterministic for tied scores so some query pairs
# may be swapped.  We sort both tensors by box coordinates before comparing.
torch.manual_seed(0)
test_input = torch.randn(1, 3, INPUT_H, INPUT_W)

with torch.no_grad():
    pt_logits, pt_boxes = wrapper(test_input)

ort_logits, ort_boxes = sess.run(None, {"pixel_values": test_input.numpy()})

# Sort by box coordinates to neutralise topk permutation differences
pt_order = np.lexsort(pt_boxes[0].numpy().T)
ort_order = np.lexsort(ort_boxes[0].T)

pt_logits_sorted = pt_logits[0].numpy()[pt_order]
ort_logits_sorted = ort_logits[0][ort_order]
pt_boxes_sorted = pt_boxes[0].numpy()[pt_order]
ort_boxes_sorted = ort_boxes[0][ort_order]

logits_diff = np.abs(pt_logits_sorted - ort_logits_sorted).max()
boxes_diff = np.abs(pt_boxes_sorted - ort_boxes_sorted).max()

print(f"Max absolute diff — logits:     {logits_diff:.2e}")
print(f"Max absolute diff — pred_boxes: {boxes_diff:.2e}")
assert logits_diff < 1e-3, f"logits mismatch! diff={logits_diff:.2e}"
assert boxes_diff < 1e-3, f"pred_boxes mismatch! diff={boxes_diff:.2e}"
print("\nVerification passed ✓")

Max absolute diff — logits:     5.72e-06
Max absolute diff — pred_boxes: 5.07e-07

Verification passed ✓


## Benchmark: ONNX vs PyTorch

Quick throughput comparison at batch size 1 (typical browser scenario).


In [6]:
import time

N_WARMUP = 5
N_RUNS = 20
x_np = test_input.numpy()

# PyTorch CPU
with torch.no_grad():
    for _ in range(N_WARMUP):
        wrapper(test_input)
t0 = time.perf_counter()
with torch.no_grad():
    for _ in range(N_RUNS):
        wrapper(test_input)
pt_ms = (time.perf_counter() - t0) / N_RUNS * 1000

# ONNX Runtime CPU
for _ in range(N_WARMUP):
    sess.run(None, {"pixel_values": x_np})
t0 = time.perf_counter()
for _ in range(N_RUNS):
    sess.run(None, {"pixel_values": x_np})
ort_ms = (time.perf_counter() - t0) / N_RUNS * 1000

import pandas as pd

display(
    pd.DataFrame(
        {"Runtime": ["PyTorch (CPU)", "ONNX Runtime (CPU)"], "Latency (ms)": [f"{pt_ms:.1f}", f"{ort_ms:.1f}"]}
    )
)


,Runtime,Latency (ms)
0,PyTorch (CPU),265.0
1,ONNX Runtime (CPU),99.4


## Push ONNX Model to HF Hub

Upload `model.onnx` to the `kaya-go/moku-v2` model repository alongside the PyTorch weights.

In [7]:
api = HfApi()

api.upload_file(
    path_or_fileobj=str(ONNX_PATH),
    path_in_repo="model.onnx",
    repo_id=HF_MODEL,
    repo_type="model",
    commit_message="feat: add ONNX export (opset 18, dynamic batch)",
)

print(f"Uploaded {ONNX_PATH.name} → {HF_MODEL}/model.onnx")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploaded moku-v2.onnx → kaya-go/moku-v2/model.onnx
